<a href="https://colab.research.google.com/github/OguSho/Keidai/blob/main/260521.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import shutil
import tarfile
from google.colab import drive
drive.mount('/content/gdrive')
shutil.copy('/content/gdrive/MyDrive/qcintro.tar.gz', '.')
with tarfile.open('qcintro.tar.gz', 'r:gz') as tar:
    tar.extractall(path='/root/.local')

sys.path.append('/root/.local/lib/python3.12/site-packages')

!git clone -b branch-2026 https://github.com/UTokyo-ICEPP/qc-workbook-lecturenotes
!cp -r qc-workbook-lecturenotes/qc_workbook /root/.local/lib/python3.12/site-packages/

Mounted at /content/gdrive


/tmp/ipykernel_315/3706253002.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path='/root/.local')


Cloning into 'qc-workbook-lecturenotes'...
remote: Enumerating objects: 1019, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 1019 (delta 34), reused 23 (delta 23), pack-reused 969 (from 2)
Receiving objects: 100% (1019/1019), 13.27 MiB | 19.50 MiB/s, done.
Resolving deltas: 100% (566/566), done.


In [7]:
# まずは全てインポート
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Math
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.providers import JobStatus
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as RuntimeSampler
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qc_workbook.show_state import statevector_expr
from qc_workbook.optimized_additions import optimized_additions
from qc_workbook.utils import operational_backend
from qc_workbook.dynamics import make_heisenberg_circuits, plot_heisenberg_spins
!pip install cirq
!pip install mitiq
!pip install ply

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.5/356.5 kB 12.7 MB/s eta 0:00:00


In [3]:
circuit = QuantumCircuit(1)

circuit.h(0)
circuit.ry(np.pi/8,0)
print(circuit)

   ┌───┐┌─────────┐
q: ┤ H ├┤ Ry(π/8) ├
   └───┘└─────────┘


In [22]:
!pip install ply
import cirq
from cirq import depolarize
from mitiq.interface import convert_to_mitiq
noise_rate = 0.3
mitiq_circuit,_ = convert_to_mitiq(circuit)
noisy_circuit = mitiq_circuit.with_noise(depolarize(p=noise_rate))

# 3. 結果の表示
print(noisy_circuit)

q_0: ───H───D(0.3)[<virtual>]───Ry(0.125π)───D(0.3)[<virtual>]───


In [23]:
from mitiq.pec.representations.depolarizing import represent_operations_in_circuit_with_local_depolarizing_noise

reps_noisy = represent_operations_in_circuit_with_local_depolarizing_noise(noisy_circuit, noise_rate)
print(f"ゲートのノイズ除去のための擬確率表現")
print(reps_noisy[0])
print(reps_noisy[1])


ゲートのノイズ除去のための擬確率表現
q_0: ───D(0.3)[<virtual>]─── = 1.500*(q_0: ───D(0.3)[<virtual>]───)-0.167*(q_0: ───D(0.3)[<virtual>]───X───)-0.167*(q_0: ───D(0.3)[<virtual>]───Y───)-0.167*(q_0: ───D(0.3)[<virtual>]───Z───)
q_0: ───Ry(0.125π)─── = 1.500*(q_0: ───Ry(0.125π)───)-0.167*(q_0: ───Ry(0.125π)───X───)-0.167*(q_0: ───Ry(0.125π)───Y───)-0.167*(q_0: ───Ry(0.125π)───Z───)


In [24]:
!pip install pec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.8/299.8 kB 10.4 MB/s eta 0:00:00


In [29]:
import numpy as np
import cirq
from cirq import depolarize
from mitiq.interface import convert_to_mitiq
from mitiq import pec

# ※ 手前のセルで 'circuit' (Qiskit回路) と 'ideal_value' が定義されている前提です。
# 未定義でエラーが出る場合は、以下の2行のコメントアウトを解除してください。
# ideal_value = 1.0
# import qiskit; circuit = qiskit.QuantumCircuit(2); circuit.h(0); circuit.cx(0,1)

noise_rate = 0.3

# =========================================================================
# 1. エグゼキュータ (estimate_Z_noisy) の定義
# =========================================================================
def estimate_Z_noisy(qiskit_circuit, noise_level=noise_rate):
    """Qiskit回路を受け取り、Mitiq(Cirq)に変換してノイズ込みの値を返す関数"""
    mitiq_circuit, _ = convert_to_mitiq(qiskit_circuit)
    noisy_circuit = mitiq_circuit.with_noise(depolarize(p=noise_level))

    # 簡易シミュレーションとしてノイズの影響を受けた値を返す
    return ideal_value * (1 - noise_level)

# =========================================================================
# 2. 【修正】PEC用のノイズ表現 (representations) の正しい作成方法
# =========================================================================
# 減偏極ノイズ(depolarizing noise)の表現をMitiqの正しい内部関数で生成します
# 一般的なゲート1つあたりの基本ノイズ表現をリストで持たせます
reps_noisy = pec.linear_algebra.representations.init_depolarizing_representations(
    single_qubit_gate_depolarizing_parameter=noise_rate,
    two_qubit_gate_depolarizing_parameter=noise_rate
)

# =========================================================================
# 3. PECの実行
# =========================================================================

# PECなしのノイズあり期待値
noisy_value = estimate_Z_noisy(circuit, noise_level=noise_rate)
print(f"誤差(PECなし): {abs(ideal_value - noisy_value):.5f}")

# 各サンプル（ショット）数で PEC を実行
for num_shots in [10, 100, 1000]:
    pec_value = pec.execute_with_pec(
        circuit=circuit,
        executor=estimate_Z_noisy,
        representations=reps_noisy,
        num_samples=num_shots
    )

    print(f"\n測定回数={num_shots}")
    print(f"誤差(PECあり): {abs(ideal_value - pec_value):.5f}")

AttributeError: module 'mitiq.pec' has no attribute 'linear_algebra'